# Unidad 10 · Grafos
## Resolución de la práctica en clase, la tarea y el mini-proyecto GPS

**Materia:** Estructuras de Datos  
**Estudiante:** Francisco Urrutia Calero  
**Fecha:** 27/10/2026

Este cuaderno adjunta los puntos **5 (práctica en clase)**, **6 (tarea)** y **7 (mini-proyecto)** de la Unidad 10. Para cada ejercicio explico qué pide, cómo lo pensé y qué prueba confirma que la solución funciona. Las pruebas son sentencias `assert`: si una celda termina sin error, el resultado es el esperado.

**Índice**
1. Preparación: librería de grafos
2. Práctica en clase (5.1 a 5.4)
3. Tarea (6.1 a 6.5)
4. Mini-proyecto: Sistema de navegación GPS

## 1. Preparación: la librería de grafos

Partimos de las clases del material de la unidad: `Grafo` (no ponderado) y `GrafoPonderado` (cada arista guarda un peso). A lo largo de los ejercicios les fui agregando métodos nuevos. Los dejé todos en una sola celda para que cada clase quede completa y coherente. Cada bloque tiene un comentario con el número de ejercicio al que pertenece, y en las secciones siguientes explico la idea de cada método y lo pruebo.

Elegí la **lista de adyacencia** (un diccionario `{vértice: [vecinos]}`) en vez de la matriz porque ocupa O(V + E) de memoria. En mapas y redes, cada vértice tiene pocas conexiones comparado con el total, así que una matriz desperdiciaría casi todo su espacio.

In [1]:
# =====================================================================
#  LIBRERÍA DE GRAFOS: Unidad 10
#  Base tomada del material; métodos nuevos marcados con su ejercicio.
# =====================================================================
from collections import deque
import heapq
import math


class Grafo:
    """Grafo no ponderado con lista de adyacencia (diccionario)."""

    def __init__(self, dirigido=False):
        self.adj = {}
        self.dirigido = dirigido

    # ---------- Operaciones básicas (material) ----------
    def agregar_vertice(self, v):
        if v not in self.adj:
            self.adj[v] = []

    def agregar_arista(self, u, v):
        self.agregar_vertice(u)
        self.agregar_vertice(v)
        self.adj[u].append(v)
        if not self.dirigido:
            self.adj[v].append(u)

    def vecinos(self, v):
        return self.adj.get(v, [])

    def mostrar(self):
        for v, vecinos in self.adj.items():
            print(f"{v} -> {vecinos}")

    def _validar_vertices(self, *vertices):
        """Lanza ValueError si algún vértice no existe en el grafo."""
        for v in vertices:
            if v not in self.adj:
                raise ValueError(f"El vértice {v!r} no existe en el grafo.")

    # ---------- Recorridos (material) ----------
    def bfs(self, inicio):
        visitados = {inicio}
        cola = deque([inicio])
        resultado = []
        while cola:
            actual = cola.popleft()
            resultado.append(actual)
            for vecino in self.vecinos(actual):
                if vecino not in visitados:
                    visitados.add(vecino)
                    cola.append(vecino)
        return resultado

    def dfs(self, inicio, visitados=None, resultado=None):
        if visitados is None:
            visitados = set()
            resultado = []
        visitados.add(inicio)
        resultado.append(inicio)
        for vecino in self.vecinos(inicio):
            if vecino not in visitados:
                self.dfs(vecino, visitados, resultado)
        return resultado

    # ---------- Auxiliar (usado en 5.1 y 5.2) ----------
    def transpuesto(self):
        """Copia del grafo con todas las aristas invertidas (u->v pasa a v->u)."""
        t = Grafo(dirigido=True)
        for v in self.adj:
            t.agregar_vertice(v)
        for u in self.adj:
            for v in self.vecinos(u):
                t.agregar_arista(v, u)
        return t

    # ---------- Ejercicio 5.2 ----------
    def es_conexo(self):
        """
        No dirigido: conexo si BFS desde cualquier vértice alcanza a todos.
        Dirigido: conexión fuerte (BFS en el grafo y en su transpuesto).
        """
        if not self.adj:
            return True  # convención: el grafo vacío es conexo
        inicio = next(iter(self.adj))
        n = len(self.adj)
        if len(self.bfs(inicio)) != n:
            return False
        if self.dirigido:
            return len(self.transpuesto().bfs(inicio)) == n
        return True

    # ---------- Ejercicio 5.3 ----------
    def camino_mas_corto_bfs(self, inicio, fin):
        """Ruta con menor número de saltos. Devuelve lista de vértices o None."""
        self._validar_vertices(inicio, fin)
        padre = {inicio: None}
        cola = deque([inicio])
        while cola:
            actual = cola.popleft()
            if actual == fin:
                break
            for vecino in self.vecinos(actual):
                if vecino not in padre:
                    padre[vecino] = actual
                    cola.append(vecino)
        if fin not in padre:
            return None
        camino = []
        nodo = fin
        while nodo is not None:
            camino.append(nodo)
            nodo = padre[nodo]
        return camino[::-1]

    # ---------- Tarea 6.1 ----------
    def dfs_iterativo(self, inicio):
        """DFS sin recursión, usando una pila explícita (lista LIFO)."""
        self._validar_vertices(inicio)
        visitados = set()
        resultado = []
        pila = [inicio]
        while pila:
            actual = pila.pop()
            if actual in visitados:
                continue
            visitados.add(actual)
            resultado.append(actual)
            # Se meten al revés para que el primer vecino quede arriba
            for vecino in reversed(self.vecinos(actual)):
                if vecino not in visitados:
                    pila.append(vecino)
        return resultado

    # ---------- Tarea 6.2 ----------
    def tiene_ciclo(self):
        """Detecta ciclos en un grafo dirigido usando DFS con tres colores."""
        if not self.dirigido:
            raise ValueError("tiene_ciclo() está pensado para grafos dirigidos.")
        BLANCO, GRIS, NEGRO = 0, 1, 2   # no visitado / en el camino actual / terminado
        color = {v: BLANCO for v in self.adj}

        def visitar(u):
            color[u] = GRIS
            for v in self.vecinos(u):
                if color[v] == GRIS:          # volvemos a un antecesor: ciclo
                    return True
                if color[v] == BLANCO and visitar(v):
                    return True
            color[u] = NEGRO
            return False

        return any(color[v] == BLANCO and visitar(v) for v in self.adj)

    # ---------- Tarea 6.5 ----------
    def grados(self):
        """Devuelve {vértice: (grado_entrada, grado_salida)}."""
        entrada = {v: 0 for v in self.adj}
        for u in self.adj:
            for v in self.vecinos(u):
                entrada[v] += 1
        return {v: (entrada[v], len(self.vecinos(v))) for v in self.adj}


class GrafoPonderado(Grafo):
    """Grafo con pesos. adj[u] guarda tuplas (vecino, peso)."""

    def agregar_arista(self, u, v, peso):
        if peso < 0:
            raise ValueError("Dijkstra no admite pesos negativos.")
        self.agregar_vertice(u)
        self.agregar_vertice(v)
        self.adj[u].append((v, peso))
        if not self.dirigido:
            self.adj[v].append((u, peso))

    def vecinos(self, v):
        """Solo los nombres de los vecinos, sin el peso (compatible con BFS/DFS)."""
        return [w for w, _ in self.adj.get(v, [])]

    def dijkstra(self, inicio):
        """Distancia mínima desde 'inicio' hasta todos los vértices."""
        self._validar_vertices(inicio)
        distancias = {v: math.inf for v in self.adj}
        distancias[inicio] = 0
        pq = [(0, inicio)]
        while pq:
            dist_actual, actual = heapq.heappop(pq)
            if dist_actual > distancias[actual]:
                continue
            for vecino, peso in self.adj[actual]:
                nueva = dist_actual + peso
                if nueva < distancias[vecino]:
                    distancias[vecino] = nueva
                    heapq.heappush(pq, (nueva, vecino))
        return distancias

    # ---------- Tarea 6.4 ----------
    def dijkstra_camino(self, origen, fin):
        """
        Devuelve (costo_minimo, [camino]) entre origen y fin.
        Si no hay camino devuelve (inf, []).
        """
        self._validar_vertices(origen, fin)
        distancias = {v: math.inf for v in self.adj}
        previo = {}
        distancias[origen] = 0
        pq = [(0, origen)]
        while pq:
            dist_actual, actual = heapq.heappop(pq)
            if dist_actual > distancias[actual]:
                continue
            if actual == fin:          # distancia ya definitiva: podemos parar
                break
            for vecino, peso in self.adj[actual]:
                nueva = dist_actual + peso
                if nueva < distancias[vecino]:
                    distancias[vecino] = nueva
                    previo[vecino] = actual
                    heapq.heappush(pq, (nueva, vecino))
        if distancias[fin] == math.inf:
            return math.inf, []
        camino = [fin]
        while camino[-1] != origen:    # reconstruir hacia atrás
            camino.append(previo[camino[-1]])
        return distancias[fin], camino[::-1]

print("Librería cargada: Grafo y GrafoPonderado listos.")

Librería cargada: Grafo y GrafoPonderado listos.


## 2. Práctica en clase

### 5.1 Árbol genealógico

**Qué pide:** un grafo que represente un árbol genealógico.

**Cómo lo pensé:** la relación *padre → hijo* tiene dirección, así que usé un grafo **dirigido** (`dirigido=True`). Cada flecha va del progenitor al descendiente. Con esto:

- Los **hijos** de una persona son sus vecinos directos (`vecinos`).
- Los **descendientes** son todo lo que se alcanza desde ella siguiendo flechas, es decir, un BFS desde esa persona.
- Los **ancestros** son lo que se alcanza siguiendo las flechas al revés. Para eso usé el grafo transpuesto (`transpuesto()`), donde todas las flechas están invertidas.

**Ojo:** en la vida real cada persona tiene dos padres, así que no es un árbol estricto sino un grafo dirigido acíclico (DAG). Por eso Juan y María apuntan a los mismos hijos.

In [2]:
arbol = Grafo(dirigido=True)
relaciones = [
    ("Juan", "Carlos"), ("María", "Carlos"),
    ("Juan", "Ana"),    ("María", "Ana"),
    ("Carlos", "Luis"), ("Carlos", "Sofía"),
    ("Ana", "Diego"),   ("Luis", "Mateo"),
]
for padre, hijo in relaciones:
    arbol.agregar_arista(padre, hijo)


def descendientes(g, persona):
    """Todos los descendientes de 'persona' (BFS desde ella, sin contarla)."""
    g._validar_vertices(persona)
    return g.bfs(persona)[1:]


def ancestros(g, persona):
    """Todos los antepasados: BFS en el grafo con las flechas invertidas."""
    g._validar_vertices(persona)
    return g.transpuesto().bfs(persona)[1:]


arbol.mostrar()
print("\nHijos de Carlos:       ", arbol.vecinos("Carlos"))
print("Descendientes de Juan: ", descendientes(arbol, "Juan"))
print("Ancestros de Mateo:    ", ancestros(arbol, "Mateo"))

assert set(arbol.vecinos("Carlos")) == {"Luis", "Sofía"}
assert set(descendientes(arbol, "Juan")) == {"Carlos", "Ana", "Luis", "Sofía", "Diego", "Mateo"}
assert set(ancestros(arbol, "Mateo")) == {"Luis", "Carlos", "Juan", "María"}
print("\n✅ Pruebas del árbol genealógico superadas")

Juan -> ['Carlos', 'Ana']
Carlos -> ['Luis', 'Sofía']
María -> ['Carlos', 'Ana']
Ana -> ['Diego']
Luis -> ['Mateo']
Sofía -> []
Diego -> []
Mateo -> []

Hijos de Carlos:        ['Luis', 'Sofía']
Descendientes de Juan:  ['Carlos', 'Ana', 'Luis', 'Sofía', 'Diego', 'Mateo']
Ancestros de Mateo:     ['Luis', 'Carlos', 'Juan', 'María']

✅ Pruebas del árbol genealógico superadas


### 5.2 Método `es_conexo()` con BFS

**Idea:** BFS visita todos los vértices alcanzables desde un punto de partida. En un grafo **no dirigido**, si el BFS desde cualquier vértice visita todos los vértices, el grafo es conexo. Si no, existe alguna "isla".

En un grafo **dirigido** la pregunta es más exigente. Pido **conexión fuerte**: desde cualquier vértice debo poder llegar a cualquier otro. Eso se comprueba con dos BFS desde el mismo vértice: uno en el grafo original y otro en el transpuesto. Si ambos alcanzan a todos, el grafo es fuertemente conexo.

**Caso borde:** el grafo vacío lo considero conexo por convención.

In [3]:
# Grafo no dirigido conexo: A - B - C - D
g1 = Grafo()
for u, v in [("A", "B"), ("B", "C"), ("C", "D")]:
    g1.agregar_arista(u, v)

# Grafo no dirigido con islas: A-B por un lado, C-D por otro, E sola
g2 = Grafo()
for u, v in [("A", "B"), ("C", "D")]:
    g2.agregar_arista(u, v)
g2.agregar_vertice("E")

# Dirigidos
ciclo_dir = Grafo(dirigido=True)              # A→B→C→A : fuertemente conexo
for u, v in [("A", "B"), ("B", "C"), ("C", "A")]:
    ciclo_dir.agregar_arista(u, v)

cadena_dir = Grafo(dirigido=True)             # A→B→C : no se puede volver
for u, v in [("A", "B"), ("B", "C")]:
    cadena_dir.agregar_arista(u, v)

print("g1 (conexo):            ", g1.es_conexo())
print("g2 (con islas):         ", g2.es_conexo())
print("ciclo dirigido A→B→C→A: ", ciclo_dir.es_conexo())
print("cadena dirigida A→B→C:  ", cadena_dir.es_conexo())

assert g1.es_conexo() is True
assert g2.es_conexo() is False
assert ciclo_dir.es_conexo() is True
assert cadena_dir.es_conexo() is False
assert Grafo().es_conexo() is True
print("\n✅ es_conexo() funciona en no dirigidos, dirigidos y casos borde")

g1 (conexo):             True
g2 (con islas):          False
ciclo dirigido A→B→C→A:  True
cadena dirigida A→B→C:   False

✅ es_conexo() funciona en no dirigidos, dirigidos y casos borde


### 5.3 Ruta más corta en saltos con BFS

**Idea:** en un grafo sin pesos, el camino con menos saltos es el primero que encuentra BFS, porque recorre el grafo por niveles. Pero no basta con saber *si* llegó: necesito la ruta. Por eso guardo, para cada vértice descubierto, **quién lo descubrió** (su `padre`). Cuando llego al destino, sigo los padres hacia atrás hasta el origen y doy vuelta la lista.

Devuelvo `None` si no hay camino, y una lista con un solo elemento si origen y destino son el mismo vértice.

In [4]:
red = Grafo()
for u, v in [("A", "B"), ("B", "C"), ("C", "D"), ("A", "E"), ("E", "D"), ("D", "F")]:
    red.agregar_arista(u, v)
red.agregar_vertice("Z")   # vértice aislado, sin conexiones

ruta = red.camino_mas_corto_bfs("A", "F")
print("A → F:", " → ".join(ruta), f"({len(ruta) - 1} saltos)")
print("A → A:", red.camino_mas_corto_bfs("A", "A"))
print("A → Z:", red.camino_mas_corto_bfs("A", "Z"))

# A-E-D-F tiene 3 saltos; A-B-C-D-F tendría 4, así que BFS debe elegir el primero
assert ruta == ["A", "E", "D", "F"]
assert red.camino_mas_corto_bfs("A", "A") == ["A"]
assert red.camino_mas_corto_bfs("A", "Z") is None
print("\n✅ Ruta mínima en saltos correcta")

A → F: A → E → D → F (3 saltos)
A → A: ['A']
A → Z: None

✅ Ruta mínima en saltos correcta


### 5.4 Simulación de vuelos con costos y Dijkstra

**Idea:** cada vuelo es una arista y su precio es el peso. Dijkstra calcula el costo mínimo desde una ciudad de origen hasta todas las demás, **incluyendo escalas**. Por eso el costo a Berlín sale 205 € por Roma y no 210 € por París: Dijkstra suma los tramos y compara.

Simplificaciones que tomé:
- Grafo **no dirigido**: asumo que el precio de ida y de vuelta es el mismo.
- Pesos **positivos**: Dijkstra no soporta pesos negativos, y `agregar_arista` lo rechaza.

Por ahora solo calculo el **costo**. El camino exacto (qué escalas hacer) lo resuelvo en el punto 6.4.

In [5]:
def construir_vuelos():
    """Red de vuelos de ejemplo: cada arista es una tarifa en euros."""
    tarifas = [
        ("Madrid", "París", 120), ("Madrid", "Roma", 95),   ("Madrid", "Lisboa", 70),
        ("París", "Londres", 80), ("París", "Berlín", 90),  ("Roma", "Berlín", 110),
        ("Lisboa", "Londres", 150), ("Berlín", "Londres", 60),
    ]
    vuelos = GrafoPonderado()
    for origen, destino, precio in tarifas:
        vuelos.agregar_arista(origen, destino, precio)
    return vuelos


vuelos = construir_vuelos()
precios = vuelos.dijkstra("Madrid")

print(f"{'Destino':<10} {'Costo mínimo':>12}")
print("-" * 24)
for ciudad in sorted(precios):
    print(f"{ciudad:<10} {precios[ciudad]:>10} €")

assert precios["Madrid"] == 0
assert precios["Lisboa"] == 70
assert precios["Londres"] == 200   # Madrid → París → Londres
assert precios["Berlín"] == 205    # Madrid → Roma → Berlín
print("\n✅ Costos mínimos desde Madrid calculados correctamente")

Destino    Costo mínimo
------------------------
Berlín            205 €
Lisboa             70 €
Londres           200 €
Madrid              0 €
París             120 €
Roma               95 €

✅ Costos mínimos desde Madrid calculados correctamente


## 3. Tarea

### 6.1 DFS iterativo

**Idea:** la versión recursiva usa la pila de llamadas del sistema. Yo hice lo mismo con una pila propia (una lista con `append` y `pop`, que funciona como LIFO).

Para que el orden coincida con la versión recursiva hay dos detalles:
1. Al meter los vecinos los recorro **al revés**, así el primer vecino queda arriba y se explora primero.
2. Marco un vértice como visitado cuando lo **saco** de la pila, no cuando lo meto, y salto los que ya estén visitados. Así un mismo vértice puede estar en la pila varias veces, pero solo se procesa una vez.

**Prueba:** primero un ejemplo a mano; luego comparo ambas versiones en 50 grafos aleatorios, con semilla fija para que el resultado sea reproducible.

In [6]:
import random

g = Grafo()
for u, v in [("A", "B"), ("A", "C"), ("B", "D"), ("C", "E"), ("D", "E")]:
    g.agregar_arista(u, v)

print("DFS recursivo:", g.dfs("A"))
print("DFS iterativo:", g.dfs_iterativo("A"))
assert g.dfs_iterativo("A") == g.dfs("A")

# Comparación con 50 grafos aleatorios (dirigidos y no dirigidos)
random.seed(42)
for _ in range(50):
    gr = Grafo(dirigido=random.random() < 0.5)
    n = random.randint(3, 10)
    for _ in range(random.randint(n, 3 * n)):
        gr.agregar_arista(random.randrange(n), random.randrange(n))
    inicio = next(iter(gr.adj))
    assert gr.dfs_iterativo(inicio) == gr.dfs(inicio)

print("\n✅ El DFS iterativo coincide con el recursivo en 50 grafos aleatorios")

DFS recursivo: ['A', 'B', 'D', 'E', 'C']
DFS iterativo: ['A', 'B', 'D', 'E', 'C']

✅ El DFS iterativo coincide con el recursivo en 50 grafos aleatorios


### 6.2 Detectar ciclos en un grafo dirigido

**Idea:** uso DFS con tres estados para cada vértice:
- **Blanco:** todavía no lo visité.
- **Gris:** está en el camino que estoy recorriendo ahora mismo (en la pila de recursión).
- **Negro:** ya terminé de explorarlo y todo lo que sale de él.

Hay ciclo si, estando en un vértice, encuentro una flecha hacia un vértice **gris**. Eso significa que volví a un antecesor del camino actual, y a eso se le llama arista de retroceso.

**¿Por qué no basta con "ya visitado"?** En un grafo dirigido, un vértice ya visitado puede ser alcanzable por otra ruta sin que haya ciclo. Por ejemplo, en un "rombo" A→B→D y A→C→D, el vértice D se visita dos veces y no hay ciclo. Por eso distingo gris (en mi camino actual) de negro (terminado).

El método solo tiene sentido en grafos dirigidos; en uno no dirigido cada arista parecería un ciclo de ida y vuelta, así que lanza `ValueError`.

In [7]:
# DAG (sin ciclos): un "rombo" donde D se alcanza por dos caminos distintos
dag = Grafo(dirigido=True)
for u, v in [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D")]:
    dag.agregar_arista(u, v)

# Ciclo de tres vértices
ciclo = Grafo(dirigido=True)
for u, v in [("A", "B"), ("B", "C"), ("C", "A")]:
    ciclo.agregar_arista(u, v)

# Ciclo de un solo vértice (autobucle)
autociclo = Grafo(dirigido=True)
autociclo.agregar_arista("X", "X")

# Dos componentes: una sin ciclo (P→Q→R) y otra con ciclo (M→N→O→N)
mixto = Grafo(dirigido=True)
for u, v in [("P", "Q"), ("Q", "R"), ("M", "N"), ("N", "O"), ("O", "N")]:
    mixto.agregar_arista(u, v)

for nombre, g_dir in [("DAG (rombo)", dag), ("ciclo A-B-C", ciclo),
                      ("autociclo", autociclo), ("mixto", mixto)]:
    print(f"{nombre:<14} → tiene ciclo: {g_dir.tiene_ciclo()}")

assert dag.tiene_ciclo() is False
assert ciclo.tiene_ciclo() is True
assert autociclo.tiene_ciclo() is True
assert mixto.tiene_ciclo() is True

# Caso de error: grafo no dirigido
try:
    no_dirigido = Grafo()
    no_dirigido.agregar_arista("A", "B")
    no_dirigido.tiene_ciclo()
    raise AssertionError("Debió lanzar ValueError")
except ValueError as e:
    print("\nError esperado en no dirigidos:", e)

print("\n✅ Detección de ciclos correcta")

DAG (rombo)    → tiene ciclo: False
ciclo A-B-C    → tiene ciclo: True
autociclo      → tiene ciclo: True
mixto          → tiene ciclo: True

Error esperado en no dirigidos: tiene_ciclo() está pensado para grafos dirigidos.

✅ Detección de ciclos correcta


### 6.4 Dijkstra que devuelve el camino exacto

Lo pongo antes del 6.3 porque la red de tráfico lo necesita.

**Idea:** es el mismo Dijkstra de la práctica, pero además guardo `previo[v]`: el vértice desde el que llegué a `v` con el costo mínimo. Cada vez que mejoro la distancia de un vértice, actualizo su previo. Al terminar, reconstruyo la ruta siguiendo `previo` desde el destino hacia el origen.

**Dos detalles importantes:**
- Corto el ciclo cuando saco el destino de la cola. En ese momento su distancia ya es definitiva, así que no hace falta seguir.
- Si el destino tiene distancia infinita, no hay camino. Devuelvo `(inf, [])` en vez de un error, para que quien llame decida qué hacer.

**Prueba:** uso la red de vuelos de la práctica (5.4) y casos borde: origen igual a destino y destino aislado.

In [8]:
vuelos = construir_vuelos()

costo, camino = vuelos.dijkstra_camino("Madrid", "Berlín")
print(f"Madrid → Berlín: {costo} € por {' → '.join(camino)}")

costo2, camino2 = vuelos.dijkstra_camino("Madrid", "Londres")
print(f"Madrid → Londres: {costo2} € por {' → '.join(camino2)}")

assert (costo, camino) == (205, ["Madrid", "Roma", "Berlín"])
assert (costo2, camino2) == (200, ["Madrid", "París", "Londres"])
assert vuelos.dijkstra_camino("Madrid", "Madrid") == (0, ["Madrid"])

# Caso sin camino: un destino aislado
aislado = construir_vuelos()
aislado.agregar_vertice("Tokio")
c, ruta_vacia = aislado.dijkstra_camino("Madrid", "Tokio")
print(f"Madrid → Tokio: costo={c}, camino={ruta_vacia}")
assert c == math.inf and ruta_vacia == []

print("\n✅ Dijkstra con reconstrucción de camino correcto")

Madrid → Berlín: 205 € por Madrid → Roma → Berlín
Madrid → Londres: 200 € por Madrid → París → Londres
Madrid → Tokio: costo=inf, camino=[]

✅ Dijkstra con reconstrucción de camino correcto


### 6.3 Red de tráfico: camino de menor latencia

**Qué pide:** un grafo ponderado que represente una red y encontrar el camino de menor latencia.

**Cómo lo pensé:** cada router es un vértice y cada enlace es una arista con su latencia en milisegundos. Usé un grafo **dirigido** porque en redes reales la latencia de A→B puede ser distinta de la de B→A (por congestión o rutas asimétricas). Ejemplo: R5 solo tiene un enlace de salida, hacia R1.

Para el camino usé `dijkstra_camino` (6.4). Verifico la ruta a mano: R1→R2→R3→R4→R5 suma 5 + 2 + 1 + 3 = 11 ms. Las otras alternativas suman más (R1→R3→R4→R5 suma 14 ms, por ejemplo).

In [9]:
red_trafico = GrafoPonderado(dirigido=True)
enlaces = [
    ("R1", "R2", 5), ("R1", "R3", 10), ("R2", "R3", 2), ("R2", "R4", 7),
    ("R3", "R4", 1), ("R3", "R5", 9),  ("R4", "R5", 3), ("R5", "R1", 4),
]
for origen, destino, ms in enlaces:
    red_trafico.agregar_arista(origen, destino, ms)

lat, ruta = red_trafico.dijkstra_camino("R1", "R5")
print(f"Menor latencia R1 → R5: {lat} ms por {' → '.join(ruta)}")

lat2, ruta2 = red_trafico.dijkstra_camino("R5", "R2")
print(f"Menor latencia R5 → R2: {lat2} ms por {' → '.join(ruta2)}")

assert (lat, ruta) == (11, ["R1", "R2", "R3", "R4", "R5"])
assert (lat2, ruta2) == (9, ["R5", "R1", "R2"])   # la vuelta no es igual a la ida

print("\n✅ Ruta de menor latencia correcta (y distinta en cada sentido)")

Menor latencia R1 → R5: 11 ms por R1 → R2 → R3 → R4 → R5
Menor latencia R5 → R2: 9 ms por R5 → R1 → R2

✅ Ruta de menor latencia correcta (y distinta en cada sentido)


### 6.5 Grados de entrada y salida

**Idea:**
- El **grado de salida** de un vértice es la cantidad de flechas que salen de él, es decir, el largo de su lista de adyacencia. Es directo.
- El **grado de entrada** no está guardado en ningún lado. Hay que contarlo recorriendo todas las listas y sumando 1 cada vez que aparece el vértice como destino.

El método recorre el grafo una vez, así que cuesta O(V + E).

**Verificación:** la suma de todos los grados de entrada y la suma de todos los grados de salida deben ser iguales al número de aristas, porque cada arista sale de un vértice y llega a otro.

In [10]:
g_dir = Grafo(dirigido=True)
for u, v in [("A", "B"), ("A", "C"), ("B", "C"), ("C", "A"), ("D", "C")]:
    g_dir.agregar_arista(u, v)

grados_g = g_dir.grados()
for v, (ent, sal) in grados_g.items():
    print(f"{v}: entrada = {ent}, salida = {sal}")

assert grados_g == {"A": (1, 2), "B": (1, 1), "C": (3, 1), "D": (0, 1)}

total_entrada = sum(e for e, _ in grados_g.values())
total_salida = sum(s for _, s in grados_g.values())
assert total_entrada == total_salida == 5   # 5 aristas en total
print("\nSuma de entradas = suma de salidas = 5 aristas ✔")

# Aplicado al árbol genealógico de 5.1: entrada = padres, salida = hijos
print("\nÁrbol genealógico (padres, hijos):")
for persona, (padres, hijos) in arbol.grados().items():
    print(f"  {persona:<8} padres={padres}  hijos={hijos}")
assert arbol.grados()["Carlos"] == (2, 2)
print("\n✅ Grados calculados correctamente")

A: entrada = 1, salida = 2
B: entrada = 1, salida = 1
C: entrada = 3, salida = 1
D: entrada = 0, salida = 1

Suma de entradas = suma de salidas = 5 aristas ✔

Árbol genealógico (padres, hijos):
  Juan     padres=0  hijos=2
  Carlos   padres=2  hijos=2
  María    padres=0  hijos=2
  Ana      padres=2  hijos=1
  Luis     padres=1  hijos=1
  Sofía    padres=1  hijos=0
  Diego    padres=1  hijos=0
  Mateo    padres=1  hijos=0

✅ Grados calculados correctamente


## 4. Mini-proyecto: Sistema de navegación GPS

**Versión:** sistema para países de Centroamérica. Cada país es un nodo y cada carretera es una arista no dirigida con su tiempo en minutos. El programa permite ver los países, agregar carreteras, consultar la ruta más rápida entre dos países y guardar una imagen del grafo con la ruta marcada en rojo.

El cálculo de la ruta usa Dijkstra con , como en la sección 5.4. Aquí el grafo se guarda en un diccionario de diccionarios () en lugar de usar la clase .

In [ ]:
import heapq, math, os, unicodedata
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# país: (lat, lon) aproximada de su capital, solo para dibujar
POS = {"Guatemala": (14.63, -90.51), "El Salvador": (13.69, -89.22), "Honduras": (14.07, -87.19),
       "Nicaragua": (12.13, -86.27), "Costa Rica": (9.93, -84.08), "Panamá": (8.98, -79.52)}
# carreteras: (origen, destino, minutos aproximados)
RUTAS = [("Guatemala", "El Salvador", 240), ("El Salvador", "Honduras", 300),
         ("Guatemala", "Honduras", 360), ("Honduras", "Nicaragua", 540),
         ("Nicaragua", "Costa Rica", 540), ("Costa Rica", "Panamá", 840)]
ALIAS = {"salvador": "El Salvador"}
CARPETA = r"C:\Users\Seong JeUC\Videos\U10\Imagen Grafo"

def clave(t):
    t = "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c))
    return " ".join(t.lower().split())

NOMBRES = {clave(n): n for n in POS} | {clave(a): v for a, v in ALIAS.items()}
ADJ = {n: {} for n in POS}
for a, b, m in RUTAS:
    ADJ[a][b] = ADJ[b][a] = m

def buscar(t):
    if clave(t) not in NOMBRES:
        raise ValueError(f"'{t}' no es un país. Usa la opción 1 para ver la lista.")
    return NOMBRES[clave(t)]

def ruta(origen, destino):
    o, d = buscar(origen), buscar(destino)
    dist, previo, cola = {n: math.inf for n in ADJ}, {}, [(0, o)]
    dist[o] = 0
    while cola:
        c, u = heapq.heappop(cola)
        if c > dist[u]:
            continue
        for v, m in ADJ[u].items():
            if c + m < dist[v]:
                dist[v], previo[v] = c + m, u
                heapq.heappush(cola, (c + m, v))
    if dist[d] == math.inf:
        raise ValueError(f"No hay carreteras entre {o} y {d}.")
    camino = [d]
    while camino[-1] != o:
        camino.append(previo[camino[-1]])
    return dist[d], camino[::-1]

def dibujar(camino=None, archivo="grafo_gps_centroamerica.png"):
    fig, ax = plt.subplots(figsize=(11, 8))
    en_ruta = set(zip(camino, camino[1:])) | set(zip(camino[1:], camino)) if camino else set()
    for a in ADJ:
        for b, m in ADJ[a].items():
            if a < b:
                r = (a, b) in en_ruta
                (la, oa), (lb, ob) = POS[a], POS[b]
                ax.plot([oa, ob], [la, lb], "#d64545" if r else "#9aa5b1", lw=4 if r else 1.5)
                ax.text((oa + ob) / 2, (la + lb) / 2, f"{m:g} min", fontsize=8, ha="center",
                        bbox=dict(fc="white", ec="none", alpha=0.85))
    for n, (la, lo) in POS.items():
        ax.scatter(lo, la, s=400, zorder=3, color="#d64545" if camino and n in camino else "#2f80ed")
        ax.text(lo, la - 0.35 if n == "El Salvador" else la + 0.3, n, fontsize=10, fontweight="bold", ha="center")
    titulo = "Carreteras entre países de Centroamérica (tiempos aproximados)"
    if camino:
        titulo += f"\nRuta: {' → '.join(camino)}"
    ax.set_title(titulo)
    ax.set_aspect("equal")
    ax.axis("off")
    try:
        os.makedirs(CARPETA, exist_ok=True)
        ruta_archivo = os.path.join(CARPETA, archivo)
        fig.savefig(ruta_archivo, dpi=150, bbox_inches="tight")
        print(f"Imagen guardada: {ruta_archivo}")
    except OSError as e:
        print(f"No se pudo guardar: {e}")
    finally:
        plt.close(fig)

if __name__ == "__main__":
    camino = None
    while True:
        print("\n1. Ver países\n2. Agregar carretera\n3. Ruta más rápida\n4. Guardar imagen\n0. Salir")
        op = input("Opción: ").strip()
        if op == "0":
            break
        try:
            if op == "1":
                print(", ".join(sorted(ADJ)))
            elif op == "2":
                a, b = buscar(input("Origen: ")), buscar(input("Destino: "))
                m = float(input("Minutos: "))
                if m <= 0 or a == b:
                    raise ValueError("Minutos positivos y países distintos.")
                ADJ[a][b] = ADJ[b][a] = m
                print("Listo.")
            elif op == "3":
                total, camino = ruta(input("Origen: "), input("Destino: "))
                print(" → ".join(camino), f"| {total:g} min")
            elif op == "4":
                dibujar(camino)
            else:
                print("?")
        except (ValueError, OSError) as e:
            print("Error:", e)
    dibujar(camino)

### Uso

Al ejecutar la celda aparece el menú en la consola de Jupyter. Escribe el número de la opción y responde las preguntas. La opción 4 guarda la imagen y la opción 0 sale, guardando también la imagen al final.

La variable  tiene una ruta de Windows. Si usas otro sistema operativo, cámbiala por una carpeta válida.